<a href="https://colab.research.google.com/github/balasri03/Mini_project/blob/main/semevalB_with_bert_bart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

LSTM+BASELINE

In [ ]:
# Basic libraries
import re
import time
import numpy as np
import pandas as pd

# PyTorch libraries (your COVID code uses these)
import torch
import torch.nn as nn
import torch.optim as optim

from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, TensorDataset

# Vocabulary
from collections import Counter

# Metrics
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
train_df = pd.read_excel("train_data_semeval2016.xlsx")

test_df = pd.read_excel("SemEval2016-Task6-subtaskB-testdata-gold.xlsx")

In [ ]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(

train_df,

test_size=0.1,

random_state=42,

stratify=train_df["Stance"]

)

In [ ]:
print(train_df.shape)

print(val_df.shape)

print(test_df.shape)

In [ ]:
print(len(train_df))
print(len(val_df))
print(len(test_df))

In [ ]:
import re

def clean_text(text):

    text = str(text).lower()

    text = re.sub(r"http\S+|www\S+", "", text)   # remove URLs

    text = re.sub(r"@\w+", "", text)             # remove mentions

    text = re.sub(r"#", "", text)                # remove hashtags

    text = re.sub(r"[^a-z\s]", "", text)        # keep only letters

    text = re.sub(r"\s+", " ", text).strip()    # remove extra spaces

    return text


train_df["clean_text"] = train_df["Tweet"].apply(clean_text)

val_df["clean_text"] = val_df["Tweet"].apply(clean_text)

test_df["clean_text"] = test_df["Tweet"].apply(clean_text)

In [ ]:
print(train_df[["Tweet", "clean_text"]].head())

In [ ]:
from collections import Counter

def build_vocab(texts, min_freq=5):

    counter = Counter()

    for text in texts:

        cleaned = clean_text(text)

        words = cleaned.split()

        counter.update(words)

    vocab = {

        "<pad>": 0,

        "<unk>": 1

    }

    for word, freq in counter.items():

        if freq >= min_freq:

            vocab[word] = len(vocab)

    return vocab

In [ ]:
def text_to_seq(text, vocab):

    words = text.split()

    return [

        vocab.get(word, vocab["<unk>"])

        for word in words

    ]

In [ ]:
train_df["input_text"] = train_df["clean_text"] + " " + train_df["Target"]

val_df["input_text"]   = val_df["clean_text"]   + " " + val_df["Target"]

test_df["input_text"]  = test_df["clean_text"]  + " " + test_df["Target"]


vocab = build_vocab(train_df["input_text"].tolist())

print(f"Vocab size: {len(vocab)}")

In [ ]:
len(train_df['input_text'])

In [ ]:
label_col = "Stance"

unique_labels = sorted(train_df[label_col].unique())

label2id = {label: idx for idx, label in enumerate(unique_labels)}

id2label = {v: k for k, v in label2id.items()}

print("Label mapping:", label2id)

In [ ]:
# Model
from tensorflow.keras.models import Sequential

# Layers
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import Embedding
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import SpatialDropout1D

# Preprocessing
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

# Callbacks
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
model = Sequential()

model.add(Embedding(input_dim=len(vocab), output_dim=100))

# model.add(SpatialDropout1D(0.4))

# model.add(LSTM(32, return_sequences=True))

# model.add(Dropout(0.4))

# model.add(LSTM(32))




model.add(SpatialDropout1D(0.5))

model.add(LSTM(32, return_sequences=True, dropout=0.3, recurrent_dropout=0.3))

model.add(Dropout(0.5))

model.add(LSTM(32, dropout=0.3, recurrent_dropout=0.3))

model.add(Dense(3, activation="softmax"))

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
X_train_seq = [text_to_seq(text, vocab) for text in train_df["input_text"]]

X_val_seq = [text_to_seq(text, vocab) for text in val_df["input_text"]]


y_train = np.array([label2id[label] for label in train_df["Stance"]])

y_val = np.array([label2id[label] for label in val_df["Stance"]])


all_sequences = X_train_seq + X_val_seq

max_sequence_length = max(len(s) for s in all_sequences)

In [ ]:
print(len(y_train))
print(len(y_val))

In [ ]:
X_train_padded = pad_sequences(
    X_train_seq,
    maxlen=max_sequence_length,
    padding='post',
    value=vocab["<pad>"]
)

X_val_padded = pad_sequences(
    X_val_seq,
    maxlen=max_sequence_length,
    padding='post',
    value=vocab["<pad>"]
)
X_test_seq = [text_to_seq(text, vocab) for text in test_df["input_text"]]

y_test = np.array([label2id[label] for label in test_df["Stance"]])
# ADD THIS for test set (important)
X_test_padded = pad_sequences(
    X_test_seq,
    maxlen=max_sequence_length,
    padding='post',
    value=vocab["<pad>"]
)


early_stop = EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True
)

In [ ]:
print(len(y_train))
print(len(y_val))

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)

class_weight_dict = dict(zip(classes, class_weights))

print(class_weight_dict)

In [ ]:

# history = model.fit(
#     X_train_padded,
#     y_train,
#     epochs=25,
#     batch_size=16,
#     validation_data=(X_val_padded, y_val),
#     callbacks=[early_stop],
#     class_weight=class_weight_dict
# )
import time

start_train = time.time()

history = model.fit(
    X_train_padded,
    y_train,
    epochs=25,
    batch_size=16,
    validation_data=(X_val_padded, y_val),
    callbacks=[early_stop],
    class_weight=class_weight_dict
)

end_train = time.time()

train_time_hours = (end_train - start_train) / 3600

print("Train Time (h):", train_time_hours)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# predictions
y_pred_labels = np.argmax(y_pred, axis=1)

# metrics
accuracy = accuracy_score(y_test, y_pred_labels)

precision = precision_score(
    y_test,
    y_pred_labels,
    average="macro"
)

recall = recall_score(
    y_test,
    y_pred_labels,
    average="macro"
)

f1 = f1_score(
    y_test,
    y_pred_labels,
    average="macro"
)

# parameters (Keras model)
params = model.count_params() / 1e6

table = pd.DataFrame({

    "Dataset": ["SemEval-A"],
    "Model": ["LSTM Baseline"],

    "Accuracy": [round(accuracy,4)],
    "Precision": [round(precision,4)],
    "Recall": [round(recall,4)],
    "F1-Score": [round(f1,4)],

    "Train Time (h)": [round(train_time_hours,3)],
    "Infer Time (s)": [round(infer_time,2)],

    "Params (M)": [round(params,2)]

})

print(table)

In [ ]:
start_infer = time.time()

y_pred = model.predict(X_test_padded)

end_infer = time.time()

infer_time = end_infer - start_infer

print("Inference Time (s):", infer_time)

In [ ]:
params = model.count_params()

params_million = params / 1e6

print("Parameters (M):", params_million)

In [ ]:
# After model training

print("Final Training Accuracy:", history.history['accuracy'][-1])
print("Final Validation Accuracy:", history.history['val_accuracy'][-1])

In [ ]:
print("Best Validation Accuracy:", max(history.history['val_accuracy']))

In [ ]:
# Convert test text to sequence
X_test_seq = [text_to_seq(text, vocab) for text in test_df["input_text"]]

# Padding
X_test_padded = pad_sequences(
    X_test_seq,
    maxlen=max_sequence_length,
    padding='post',
    value=vocab["<pad>"]
)

# Prediction
prediction = model.predict(X_test_padded)

In [ ]:
import numpy as np

y_pred = np.argmax(prediction, axis=1)

In [ ]:
y_test = np.array([label2id[label] for label in test_df["Stance"]])

In [ ]:
from sklearn.metrics import confusion_matrix

val_predictions = model.predict(X_val_padded)

y_val_pred = np.argmax(val_predictions, axis=1)

cm_val = confusion_matrix(y_val, y_val_pred)

print("Validation Confusion Matrix:")
print(cm_val)

In [ ]:
test_predictions = model.predict(X_test_padded)

y_test_pred = np.argmax(test_predictions, axis=1)

cm_test = confusion_matrix(y_test, y_test_pred)

print("Test Confusion Matrix:")
print(cm_test)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

# Step 1: Get prediction probabilities
y_pred_prob = model.predict(X_test_padded)

# Step 2: Convert test labels to binary format
y_test_bin = label_binarize(y_test, classes=[0,1,2])

n_classes = 3

fpr = dict()
tpr = dict()
roc_auc = dict()

# Step 3: Compute ROC for each class
for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_pred_prob[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Step 4: Plot ROC
plt.figure(figsize=(6,6))

colors = ['blue','red','green']
labels = ['AGAINST','FAVOR','NONE']

for i in range(n_classes):
    plt.plot(
        fpr[i],
        tpr[i],
        color=colors[i],
        label=f"{labels[i]} (AUC = {roc_auc[i]:.2f})"
    )

plt.plot([0,1],[0,1],'k--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Baseline LSTM")
plt.legend()
plt.show()

ONE HOT ENCODING

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Lambda, Dropout
from tensorflow.keras.regularizers import l2

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Lambda
from tensorflow.keras.regularizers import l2

vocab_size = len(vocab)

model_ohe = Sequential()

# One-Hot Encoding Layer
model_ohe.add(
    Lambda(
        lambda x: tf.one_hot(tf.cast(x, tf.int32), depth=vocab_size),
        input_shape=(max_sequence_length,)
    )
)

# First LSTM
model_ohe.add(
    LSTM(
        24,
        return_sequences=True,
        kernel_regularizer=l2(0.0005),
        recurrent_dropout=0.2
    )
)

model_ohe.add(Dropout(0.4))

# Second LSTM
model_ohe.add(
    LSTM(
        24,
        kernel_regularizer=l2(0.0005),
        recurrent_dropout=0.2
    )
)

model_ohe.add(Dropout(0.4))

# Output
model_ohe.add(Dense(3, activation="softmax"))

In [ ]:
model_ohe.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(zip(np.unique(y_train), class_weights))

print(class_weights)

In [ ]:
# # increase NONE class weight manually
# class_weights[2] = class_weights[2] * 1.5   # try 1.5 first

# print(class_weights)

In [ ]:
# reset from original first (important)
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(zip(np.unique(y_train), weights))

# now boost both weak classes
class_weights[1] = class_weights[1] * 1.5   # FAVOR
class_weights[2] = class_weights[2] * 1.5   # NONE

print(class_weights)

In [ ]:
import time

start_train = time.time()

history = model.fit(
    X_train_padded,
    y_train,
    epochs=25,
    batch_size=16,
    validation_data=(X_val_padded, y_val),
    callbacks=[early_stop],
    class_weight=class_weight_dict
)

end_train = time.time()

train_time_hours = (end_train - start_train) / 3600

In [ ]:
start_infer = time.time()

y_pred = model.predict(X_test_padded)

end_infer = time.time()

infer_time = end_infer - start_infer

In [ ]:
import numpy as np

y_pred_labels = np.argmax(y_pred, axis=1)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy = accuracy_score(y_test, y_pred_labels)

precision = precision_score(
    y_test,
    y_pred_labels,
    average="macro"
)

recall = recall_score(
    y_test,
    y_pred_labels,
    average="macro"
)

f1 = f1_score(
    y_test,
    y_pred_labels,
    average="macro"
)

In [ ]:
params = model.count_params() / 1e6

In [ ]:
print("\n===== LSTM ONE-HOT MODEL RESULTS =====")

print("Accuracy:", round(accuracy,4))
print("Precision:", round(precision,4))
print("Recall:", round(recall,4))
print("F1 Score:", round(f1,4))

print("Train Time (h):", round(train_time_hours,3))
print("Infer Time (s):", round(infer_time,2))

print("Params (M):", round(params,2))

In [ ]:
import numpy as np

y_pred_prob = model.predict(X_test_padded)

In [ ]:
from sklearn.preprocessing import label_binarize

y_test_bin = label_binarize(y_test, classes=[0,1,2])

n_classes = 3

In [ ]:
from sklearn.metrics import roc_curve, auc

fpr = {}
tpr = {}
roc_auc = {}

for i in range(n_classes):

    fpr[i], tpr[i], _ = roc_curve(
        y_test_bin[:, i],
        y_pred_prob[:, i]
    )

    roc_auc[i] = auc(fpr[i], tpr[i])

In [ ]:
from sklearn.metrics import roc_curve, auc

fpr = {}
tpr = {}
roc_auc = {}

for i in range(n_classes):

    fpr[i], tpr[i], _ = roc_curve(
        y_test_bin[:, i],
        y_pred_prob[:, i]
    )

    roc_auc[i] = auc(fpr[i], tpr[i])

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,6))

colors = ['blue','red','green']
labels = ['AGAINST','FAVOR','NONE']

for i in range(n_classes):

    plt.plot(
        fpr[i],
        tpr[i],
        color=colors[i],
        label=f"{labels[i]} (AUC = {roc_auc[i]:.3f})"
    )

plt.plot([0,1],[0,1],'k--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.title("ROC Curve – LSTM OneHot")

plt.legend()
plt.show()

In [ ]:
val_predictions = model_ohe.predict(X_val_padded)

y_val_pred_ohe = np.argmax(val_predictions, axis=1)

# Confusion matrix
cm = confusion_matrix(y_val, y_val_pred_ohe)

print("Confusion Matrix:")
print(cm)

GLOVE EMBEDDINGS


In [ ]:
!pip install gensim

import gensim.downloader as api

glove_model = api.load("glove-wiki-gigaword-100")

In [ ]:
vector = glove_model["vaccine"]

print(vector.shape)

In [ ]:
vocab_size = len(vocab)
embedding_dim = 100

In [ ]:
vocab_size

In [ ]:
import numpy as np

embedding_matrix = np.zeros((vocab_size, embedding_dim))

hits = 0
misses = 0

for word, idx in vocab.items():

    if word in glove_model:
        embedding_matrix[idx] = glove_model[word]
        hits += 1
    else:
        embedding_matrix[idx] = np.random.normal(scale=0.6, size=(embedding_dim,))
        misses += 1

print("Converted:", hits)
print("Missed:", misses)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, SpatialDropout1D
from tensorflow.keras.optimizers import Adam

model_glove = Sequential()

model_glove.add(
    Embedding(
        vocab_size,
        embedding_dim,
        weights=[embedding_matrix],
        # input_length=max_sequence_length,
        trainable=True   # ✅ keep TRUE for SemEval
    )
)

model_glove.add(SpatialDropout1D(0.3))

model_glove.add(LSTM(64, return_sequences=True))

model_glove.add(Dropout(0.3))

model_glove.add(LSTM(32))

model_glove.add(Dropout(0.3))

model_glove.add(Dense(3, activation="softmax"))

model_glove.compile(
    optimizer=Adam(learning_rate=0.001),   # ✅ important
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(zip(np.unique(y_train), weights))

In [ ]:
import time

# TRAIN TIME
start_train = time.time()

history_glove = model_glove.fit(
    X_train_padded,
    y_train,
    validation_data=(X_val_padded, y_val),
    epochs=25,
    batch_size=16,
    callbacks=[early_stop],
    class_weight=class_weights
)

end_train = time.time()

train_time_hours = (end_train - start_train) / 3600

In [ ]:
import numpy as np

start_infer = time.time()

y_pred_glove = model_glove.predict(X_test_padded)

end_infer = time.time()

infer_time = end_infer - start_infer

# Convert probabilities → class labels
y_pred_labels = np.argmax(y_pred_glove, axis=1)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy = accuracy_score(y_test, y_pred_labels)

precision = precision_score(
    y_test,
    y_pred_labels,
    average="macro"
)

recall = recall_score(
    y_test,
    y_pred_labels,
    average="macro"
)

f1 = f1_score(
    y_test,
    y_pred_labels,
    average="macro"
)

In [ ]:
params = model_glove.count_params() / 1e6

In [ ]:
print("\n===== LSTM + GloVe RESULTS =====")

print("Accuracy:", round(accuracy,4))
print("Precision:", round(precision,4))
print("Recall:", round(recall,4))
print("F1 Score:", round(f1,4))

print("Train Time (h):", round(train_time_hours,3))
print("Infer Time (s):", round(infer_time,2))

print("Params (M):", round(params,2))

In [ ]:
import pandas as pd

table = pd.DataFrame({

    "Dataset": ["SemEval-A"],
    "Model": ["LSTM + GloVe"],

    "Accuracy": [round(accuracy,4)],
    "Precision": [round(precision,4)],
    "Recall": [round(recall,4)],
    "F1-Score": [round(f1,4)],

    "Train Time (h)": [round(train_time_hours,3)],
    "Infer Time (s)": [round(infer_time,2)],

    "Params (M)": [round(params,2)]

})

print(table)

In [ ]:
pred = model_glove.predict(X_val_padded)

In [ ]:
from sklearn.preprocessing import label_binarize

y_val_bin = label_binarize(y_val, classes=[0,1,2])
n_classes = 3

In [ ]:
from sklearn.metrics import roc_curve, auc

fpr = {}
tpr = {}
roc_auc = {}

for i in range(n_classes):

    fpr[i], tpr[i], _ = roc_curve(
        y_val_bin[:, i],
        pred[:, i]
    )

    roc_auc[i] = auc(fpr[i], tpr[i])

In [ ]:
import matplotlib.pyplot as plt

plt.figure()

colors = ['blue','red','green']
class_names = ['AGAINST','FAVOR','NONE']

for i in range(n_classes):

    plt.plot(
        fpr[i],
        tpr[i],
        color=colors[i],
        label=f'{class_names[i]} (AUC = {roc_auc[i]:.3f})'
    )

plt.plot([0,1],[0,1],'k--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.title("ROC Curve – LSTM + GloVe")

plt.legend()

plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix

pred = model_glove.predict(X_val_padded)

y_pred = np.argmax(pred, axis=1)

cm = confusion_matrix(y_val, y_pred)

print(cm)

LSTM+GLOVE+ATTENTION

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Layer

class AttentionLayer(Layer):

    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):

        self.W = self.add_weight(
            shape=(input_shape[-1], input_shape[-1]),
            initializer="glorot_uniform",
            trainable=True)

        self.b = self.add_weight(
            shape=(input_shape[-1],),
            initializer="zeros",
            trainable=True)

        self.u = self.add_weight(
            shape=(input_shape[-1],),
            initializer="glorot_uniform",
            trainable=True)

    def call(self, inputs):

        score = tf.tanh(tf.tensordot(inputs, self.W, axes=1) + self.b)

        attention_weights = tf.nn.softmax(
            tf.tensordot(score, self.u, axes=1),
            axis=1)

        context_vector = attention_weights[..., tf.newaxis] * inputs

        context_vector = tf.reduce_sum(context_vector, axis=1)

        return context_vector

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, SpatialDropout1D

model_attention = Sequential()

# GloVe Embedding Layer
model_attention.add(
    Embedding(
        vocab_size,
        embedding_dim,
        weights=[embedding_matrix],
        input_length=max_sequence_length,
        trainable=True
    )
)

# Dropout for embedding
model_attention.add(SpatialDropout1D(0.3))

# LSTM layer
model_attention.add(
    LSTM(
        64,
        return_sequences=True
    )
)

# Attention Layer
model_attention.add(AttentionLayer())

# Dropout
model_attention.add(Dropout(0.5))

# Output Layer
model_attention.add(Dense(3, activation="softmax"))

# Compile model
model_attention.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

In [ ]:
# # class_weights = {0:2.0, 1:1.0, 2:1.0}
# class_weights = {
#     0: 2.0,   # AGAINST (keep same)
#     1: 1.8,   # FAVOR  ← increase this
#     2: 1.2    # NONE   ← slight increase
# }

# print(class_weights)

# early_stop = tf.keras.callbacks.EarlyStopping(
#     monitor="val_accuracy",
#     patience=3,
#     restore_best_weights=True
# )

# history = model_attention.fit(
#     X_train_padded,
#     y_train,
#     validation_data=(X_val_padded, y_val),
#     epochs=25,
#     batch_size=16,
#     callbacks=[early_stop],
#     class_weight=class_weights
# )
import time

start_train = time.time()

history = model_attention.fit(
    X_train_padded,
    y_train,
    validation_data=(X_val_padded, y_val),
    epochs=25,
    batch_size=16,
    callbacks=[early_stop],
    class_weight=class_weights
)

end_train = time.time()

train_time_hours = (end_train - start_train) / 3600

In [ ]:
start_infer = time.time()

y_pred_attention = model_attention.predict(X_test_padded)

end_infer = time.time()

infer_time = end_infer - start_infer

In [ ]:
import numpy as np

y_pred_labels = np.argmax(y_pred_attention, axis=1)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy = accuracy_score(y_test, y_pred_labels)

precision = precision_score(
    y_test,
    y_pred_labels,
    average="macro"
)

recall = recall_score(
    y_test,
    y_pred_labels,
    average="macro"
)

f1 = f1_score(
    y_test,
    y_pred_labels,
    average="macro"
)

In [ ]:
params = model_attention.count_params() / 1e6

In [ ]:
print("\n===== LSTM + GloVe + Attention RESULTS =====")

print("Accuracy:", round(accuracy,4))
print("Precision:", round(precision,4))
print("Recall:", round(recall,4))
print("F1 Score:", round(f1,4))

print("Train Time (h):", round(train_time_hours,3))
print("Infer Time (s):", round(infer_time,2))

print("Params (M):", round(params,2))

In [ ]:
import pandas as pd

table = pd.DataFrame({

    "Dataset": ["SemEval-A"],
    "Model": ["LSTM + GloVe + Attention"],

    "Accuracy": [round(accuracy,4)],
    "Precision": [round(precision,4)],
    "Recall": [round(recall,4)],
    "F1-Score": [round(f1,4)],

    "Train Time (h)": [round(train_time_hours,3)],
    "Infer Time (s)": [round(infer_time,2)],

    "Params (M)": [round(params,2)]

})

print(table)

In [ ]:
from sklearn.metrics import confusion_matrix

pred = model_attention.predict(X_val_padded)

y_pred = pred.argmax(axis=1)

print(confusion_matrix(y_val, y_pred))

In [ ]:
from sklearn.preprocessing import label_binarize

# Binarize labels
y_val_bin = label_binarize(y_val, classes=[0,1,2])

In [ ]:
from sklearn.metrics import roc_curve, auc

n_classes = 3

fpr = dict()
tpr = dict()
roc_auc = dict()

for i in range(n_classes):

    fpr[i], tpr[i], _ = roc_curve(y_val_bin[:, i], pred[:, i])

    roc_auc[i] = auc(fpr[i], tpr[i])

In [ ]:
import matplotlib.pyplot as plt

plt.figure()

colors = ['blue', 'red', 'green']
class_names = ['AGAINST', 'FAVOR', 'NONE']

for i in range(n_classes):

    plt.plot(
        fpr[i],
        tpr[i],
        color=colors[i],
        lw=2,
        label=f'{class_names[i]} (AUC = {roc_auc[i]:.2f})'
    )

# diagonal line
plt.plot([0,1], [0,1], 'k--')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')

plt.title('ROC Curve - LSTM + Attention')

plt.legend()

plt.show()

BASELINE GRU

In [ ]:

print("Train input_text sample:")
print(train_df["input_text"].head())

print("\nValidation input_text sample:")
print(val_df["input_text"].head())

print("\nTest input_text sample:")
print(test_df["input_text"].head())

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

# create tokenizer
tokenizer = Tokenizer(oov_token="<OOV>")

# fit on training text ONLY
tokenizer.fit_on_texts(train_df["input_text"])

# check vocab size
print("Vocab size:", len(tokenizer.word_index))

# show first sequence example
seq_example = tokenizer.texts_to_sequences([train_df["input_text"].iloc[0]])

print("\nExample text:")
print(train_df["input_text"].iloc[0])

print("\nConverted sequence:")
print(seq_example)

In [ ]:
# convert text to sequences

X_train_seq = tokenizer.texts_to_sequences(train_df["input_text"])

X_val_seq   = tokenizer.texts_to_sequences(val_df["input_text"])

X_test_seq  = tokenizer.texts_to_sequences(test_df["input_text"])


# print example

print("Train sequence example:")
print(X_train_seq[0])

print("\nValidation sequence example:")
print(X_val_seq[0])

print("\nTest sequence example:")
print(X_test_seq[0])

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

# set max length
max_len = 40

# pad sequences

X_train_padded = pad_sequences(X_train_seq, maxlen=max_len, padding='post')

X_val_padded   = pad_sequences(X_val_seq, maxlen=max_len, padding='post')

X_test_padded  = pad_sequences(X_test_seq, maxlen=max_len, padding='post')


# check shapes

print("Train padded shape:", X_train_padded.shape)

print("Validation padded shape:", X_val_padded.shape)

print("Test padded shape:", X_test_padded.shape)


# show example padded sequence

print("\nExample padded sequence:")

print(X_train_padded[0])

In [ ]:
# label mapping

label_map = {
    "FAVOR": 0,
    "AGAINST": 1,
    "NONE": 2
}

y_train = train_df["Stance"].map(label_map).values

y_val   = val_df["Stance"].map(label_map).values

y_test  = test_df["Stance"].map(label_map).values


# verify labels

print("Train labels:", set(y_train))

print("Validation labels:", set(y_val))

print("Test labels:", set(y_test))

In [ ]:
# from tensorflow.keras.models import Sequential
# from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout
# from tensorflow.keras.optimizers import Adam

# # correct vocab size
# vocab_size = len(tokenizer.word_index) + 1

# model = Sequential()

# model.add(Embedding(
#     input_dim=vocab_size,
#     output_dim=100,
#     input_length=40
# ))

# model.add(GRU(32))

# model.add(Dropout(0.5))

# model.add(Dense(3, activation='softmax'))

# model.compile(
#     optimizer=Adam(learning_rate=0.005),
#     loss='sparse_categorical_crossentropy',
#     metrics=['accuracy']
# )

# model.summary()
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout
from tensorflow.keras.optimizers import Adam

vocab_size = len(tokenizer.word_index) + 1

model_gru = Sequential()

model_gru.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=100,
        input_length=40
    )
)

model_gru.add(GRU(32))

model_gru.add(Dropout(0.5))

model_gru.add(Dense(3, activation='softmax'))

model_gru.compile(
    optimizer=Adam(learning_rate=0.005),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model_gru.summary()

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# compute class weights
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(enumerate(class_weights))

print("Class weights:", class_weights)

In [ ]:
# history = model.fit(
#     X_train_padded,
#     y_train,
#     validation_data=(X_val_padded, y_val),
#     epochs=5,
#     batch_size=32,
#     class_weight=class_weights
# )
import time

start_train = time.time()

history_gru = model_gru.fit(
    X_train_padded,
    y_train,
    validation_data=(X_val_padded, y_val),
    epochs=25,
    batch_size=16,
    callbacks=[early_stop],
    class_weight=class_weights
)

end_train = time.time()

train_time_hours = (end_train - start_train) / 3600

In [ ]:
start_infer = time.time()

y_pred_gru = model_gru.predict(X_test_padded)

end_infer = time.time()

infer_time = end_infer - start_infer

In [ ]:
import numpy as np

y_pred_labels = np.argmax(y_pred_gru, axis=1)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy = accuracy_score(y_test, y_pred_labels)

precision = precision_score(
    y_test,
    y_pred_labels,
    average="macro"
)

recall = recall_score(
    y_test,
    y_pred_labels,
    average="macro"
)

f1 = f1_score(
    y_test,
    y_pred_labels,
    average="macro"
)

In [ ]:
params = model_gru.count_params() / 1e6

In [ ]:
print("\n===== GRU BASELINE RESULTS =====")

print("Accuracy:", round(accuracy,4))
print("Precision:", round(precision,4))
print("Recall:", round(recall,4))
print("F1 Score:", round(f1,4))

print("Train Time (h):", round(train_time_hours,3))
print("Infer Time (s):", round(infer_time,2))

print("Params (M):", round(params,2))

In [ ]:
import pandas as pd

table = pd.DataFrame({

    "Dataset": ["SemEval-A"],
    "Model": ["GRU Baseline"],

    "Accuracy": [round(accuracy,4)],
    "Precision": [round(precision,4)],
    "Recall": [round(recall,4)],
    "F1-Score": [round(f1,4)],

    "Train Time (h)": [round(train_time_hours,3)],
    "Infer Time (s)": [round(infer_time,2)],

    "Params (M)": [round(params,2)]

})

print(table)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

# Step 1: Get prediction probabilities
y_pred_prob = model.predict(X_test_padded)

# Step 2: Convert test labels to binary format
y_test_bin = label_binarize(y_test, classes=[0,1,2])

n_classes = 3

fpr = {}
tpr = {}
roc_auc = {}

# Step 3: Compute ROC for each class
for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_pred_prob[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Step 4: Plot ROC curve
plt.figure(figsize=(6,6))

colors = ['blue','red','green']
labels = ['AGAINST','FAVOR','NONE']

for i in range(n_classes):
    plt.plot(
        fpr[i],
        tpr[i],
        color=colors[i],
        label=f"{labels[i]} (AUC = {roc_auc[i]:.2f})"
    )

plt.plot([0,1],[0,1],'k--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Baseline GRU")
plt.legend()
plt.show()

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report



print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

GRU+ONEHOT

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout
from tensorflow.keras.optimizers import Adam

vocab_size = len(tokenizer.word_index) + 1

model_onehot = Sequential()

# One-Hot style embedding (trainable, random init)
model_onehot.add(Embedding(
    input_dim=vocab_size,
    output_dim=vocab_size   # key difference
))

model_onehot.add(GRU(64))

model_onehot.add(Dropout(0.5))

model_onehot.add(Dense(3, activation='softmax'))

model_onehot.compile(
    optimizer=Adam(learning_rate=0.0005),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model_onehot.summary()

In [ ]:
# history_onehot = model_onehot.fit(
#     X_train_padded,
#     y_train,
#     validation_data=(X_val_padded, y_val),
#     epochs=4,
#     batch_size=32
# )
import time

start_train = time.time()

history_onehot = model_onehot.fit(
    X_train_padded,
    y_train,
    validation_data=(X_val_padded, y_val),
    epochs=25,
    batch_size=16,
    callbacks=[early_stop],
    class_weight=class_weights
)

end_train = time.time()

train_time_hours = (end_train - start_train) / 3600

In [ ]:
start_infer = time.time()

y_pred_onehot = model_onehot.predict(X_test_padded)

end_infer = time.time()

infer_time = end_infer - start_infer

In [ ]:
import numpy as np

y_pred_labels = np.argmax(y_pred_onehot, axis=1)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy = accuracy_score(y_test, y_pred_labels)

precision = precision_score(
    y_test,
    y_pred_labels,
    average="macro"
)

recall = recall_score(
    y_test,
    y_pred_labels,
    average="macro"
)

f1 = f1_score(
    y_test,
    y_pred_labels,
    average="macro"
)

In [ ]:
params = model_onehot.count_params() / 1e6

In [ ]:
print("\n===== GRU + OneHot RESULTS =====")

print("Accuracy:", round(accuracy,4))
print("Precision:", round(precision,4))
print("Recall:", round(recall,4))
print("F1 Score:", round(f1,4))

print("Train Time (h):", round(train_time_hours,3))
print("Infer Time (s):", round(infer_time,2))

print("Params (M):", round(params,2))

In [ ]:
import pandas as pd

table = pd.DataFrame({

    "Dataset": ["SemEval-A"],
    "Model": ["GRU + OneHot"],

    "Accuracy": [round(accuracy,4)],
    "Precision": [round(precision,4)],
    "Recall": [round(recall,4)],
    "F1-Score": [round(f1,4)],

    "Train Time (h)": [round(train_time_hours,3)],
    "Infer Time (s)": [round(infer_time,2)],

    "Params (M)": [round(params,2)]

})

print(table)

In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

y_pred_prob = model_onehot.predict(X_test_padded)

y_test_bin = label_binarize(y_test, classes=[0,1,2])

n_classes = 3

fpr = {}
tpr = {}
roc_auc = {}

for i in range(n_classes):

    fpr[i], tpr[i], _ = roc_curve(
        y_test_bin[:, i],
        y_pred_prob[:, i]
    )

    roc_auc[i] = auc(fpr[i], tpr[i])

plt.figure(figsize=(6,6))

colors = ['blue','red','green']
labels = ['AGAINST','FAVOR','NONE']

for i in range(n_classes):

    plt.plot(
        fpr[i],
        tpr[i],
        color=colors[i],
        label=f"{labels[i]} (AUC = {roc_auc[i]:.3f})"
    )

plt.plot([0,1],[0,1],'k--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve – GRU + OneHot")

plt.legend()
plt.show()

GLOVE EMBEDDINGS

In [ ]:
import gensim.downloader as api
glove_model = api.load("glove-wiki-gigaword-100")

In [ ]:
# embedding_dim = 100
# vocab_size = len(vocab)

# embedding_matrix = np.zeros((vocab_size, embedding_dim))

# for word, idx in vocab.items():

#     if word in glove_model:
#         embedding_matrix[idx] = glove_model[word]

#     else:
#         embedding_matrix[idx] = np.random.normal(scale=0.6, size=(embedding_dim,))
embedding_dim = 100
vocab_size = len(tokenizer.word_index) + 1

embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, idx in tokenizer.word_index.items():

    if word in glove_model:
        embedding_matrix[idx] = glove_model[word]

    else:
        embedding_matrix[idx] = np.random.normal(scale=0.6, size=(embedding_dim,))

In [ ]:
# from tensorflow.keras.models import Sequential
# from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout, SpatialDropout1D, Bidirectional
# from tensorflow.keras.optimizers import Adam

# model_gru_glove_improved = Sequential()

# model_gru_glove_improved.add(
#     Embedding(
#         input_dim=vocab_size,
#         output_dim=embedding_dim,
#         weights=[embedding_matrix],
#         input_length=max_sequence_length,
#         trainable=True
#     )
# )

# model_gru_glove_improved.add(SpatialDropout1D(0.4))

# model_gru_glove_improved.add(
#     Bidirectional(
#         GRU(96, return_sequences=True)
#     )
# )

# model_gru_glove_improved.add(Dropout(0.4))

# model_gru_glove_improved.add(
#     Bidirectional(
#         GRU(48)
#     )
# )

# model_gru_glove_improved.add(Dense(3, activation="softmax"))
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout, SpatialDropout1D, Bidirectional
from tensorflow.keras.optimizers import Adam

model_gru_glove_improved = Sequential()

model_gru_glove_improved.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        weights=[embedding_matrix],
        input_length=max_sequence_length,
        trainable=True
    )
)

model_gru_glove_improved.add(SpatialDropout1D(0.4))

model_gru_glove_improved.add(
    Bidirectional(
        GRU(96, return_sequences=True)
    )
)

model_gru_glove_improved.add(Dropout(0.4))

model_gru_glove_improved.add(
    Bidirectional(
        GRU(48)
    )
)

model_gru_glove_improved.add(Dense(3, activation="softmax"))

model_gru_glove_improved.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model_gru_glove_improved.summary()

In [ ]:
print("Tokenizer vocab:", vocab_size)
print("Embedding matrix:", embedding_matrix.shape)

In [ ]:
from sklearn.utils import resample
import pandas as pd

train_combined = pd.DataFrame({
    "text": train_df["input_text"],
    "label": y_train
})

df_against = train_combined[train_combined.label == 0]
df_favor   = train_combined[train_combined.label == 1]
df_none    = train_combined[train_combined.label == 2]

# Upsample minority classes
df_against_up = resample(df_against, replace=True, n_samples=len(df_none), random_state=42)
df_favor_up   = resample(df_favor, replace=True, n_samples=len(df_none), random_state=42)

balanced_df = pd.concat([df_none, df_against_up, df_favor_up])

print(balanced_df.label.value_counts())

In [ ]:
# model_gru_glove_improved.compile(
#     optimizer=Adam(learning_rate=0.0005),
#     loss="sparse_categorical_crossentropy",
#     metrics=["accuracy"]
# )
from tensorflow.keras.optimizers import Adam

model_gru_glove_improved.compile(
    optimizer=Adam(learning_rate=0.0005),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
class_weight_dict = {
    0: 2.0,   # AGAINST
    1: 3.5,   # FAVOR (increase strongly)
    2: 1.0    # NONE
}

In [ ]:
# history_gru_glove_improved = model_gru_glove_improved.fit(

#     X_train_padded,
#     y_train,

#     validation_data=(X_val_padded, y_val),

#     epochs=25,
#     batch_size=32,

#     class_weight=class_weight_dict,

#     callbacks=[early_stop]

# )
# history_gru_glove_improved = model_gru_glove_improved.fit(
#     X_train_padded,
#     y_train,
#     validation_data=(X_val_padded, y_val),
#     epochs=25,
#     batch_size=32,
#     class_weight=class_weight_dict,
#     callbacks=[early_stop]
# )import time

start_train = time.time()

history_gru_glove_improved = model_gru_glove_improved.fit(
    X_train_padded,
    y_train,
    validation_data=(X_val_padded, y_val),
    epochs=25,
    batch_size=32,
    class_weight=class_weight_dict,
    callbacks=[early_stop]
)

end_train = time.time()

train_time_hours = (end_train - start_train) / 3600

In [ ]:
start_infer = time.time()

y_pred_gru_glove = model_gru_glove_improved.predict(X_test_padded)

end_infer = time.time()

infer_time = end_infer - start_infer

In [ ]:
import numpy as np

y_pred_labels = np.argmax(y_pred_gru_glove, axis=1)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy = accuracy_score(y_test, y_pred_labels)

precision = precision_score(
    y_test,
    y_pred_labels,
    average="macro"
)

recall = recall_score(
    y_test,
    y_pred_labels,
    average="macro"
)

f1 = f1_score(
    y_test,
    y_pred_labels,
    average="macro"
)

In [ ]:
params = model_gru_glove_improved.count_params() / 1e6

In [ ]:
print("\n===== GRU + GloVe RESULTS =====")

print("Accuracy:", round(accuracy,4))
print("Precision:", round(precision,4))
print("Recall:", round(recall,4))
print("F1 Score:", round(f1,4))

print("Train Time (h):", round(train_time_hours,3))
print("Infer Time (s):", round(infer_time,2))

print("Params (M):", round(params,2))

In [ ]:
import pandas as pd

table = pd.DataFrame({

    "Dataset": ["SemEval-A"],
    "Model": ["GRU + GloVe"],

    "Accuracy": [round(accuracy,4)],
    "Precision": [round(precision,4)],
    "Recall": [round(recall,4)],
    "F1-Score": [round(f1,4)],

    "Train Time (h)": [round(train_time_hours,3)],
    "Infer Time (s)": [round(infer_time,2)],

    "Params (M)": [round(params,2)]

})

print(table)

In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

y_pred_prob = model_gru_glove_improved.predict(X_test_padded)

y_test_bin = label_binarize(y_test, classes=[0,1,2])

n_classes = 3

fpr = {}
tpr = {}
roc_auc = {}

for i in range(n_classes):

    fpr[i], tpr[i], _ = roc_curve(
        y_test_bin[:, i],
        y_pred_prob[:, i]
    )

    roc_auc[i] = auc(fpr[i], tpr[i])

plt.figure(figsize=(6,6))

colors = ['blue','red','green']
labels = ['AGAINST','FAVOR','NONE']

for i in range(n_classes):

    plt.plot(
        fpr[i],
        tpr[i],
        color=colors[i],
        label=f"{labels[i]} (AUC = {roc_auc[i]:.3f})"
    )

plt.plot([0,1],[0,1],'k--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.title("ROC Curve – GRU + GloVe")

plt.legend()
plt.show()

In [ ]:
y_probs = model_gru_glove_improved.predict(X_test_padded)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

# Step 1: Get prediction probabilities
y_pred_prob = model_gru_glove_improved.predict(X_test_padded)

# Step 2: Convert test labels to binary
y_test_bin = label_binarize(y_test, classes=[0,1,2])

n_classes = 3

fpr = {}
tpr = {}
roc_auc = {}

# Step 3: Compute ROC for each class
for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_pred_prob[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Step 4: Plot ROC curves
plt.figure(figsize=(6,6))

labels = ["AGAINST","FAVOR","NONE"]
colors = ["blue","red","green"]

for i in range(n_classes):
    plt.plot(
        fpr[i],
        tpr[i],
        color=colors[i],
        label=f"{labels[i]} (AUC = {roc_auc[i]:.3f})"
    )

plt.plot([0,1],[0,1],'k--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - GRU + GloVe Improved")
plt.legend()
plt.show()

In [ ]:
import numpy as np

def predict_with_thresholds(probs, favor_th, against_th):

    preds = []

    for p in probs:

        against = p[0]
        favor   = p[1]
        none    = p[2]

        if favor > favor_th:
            preds.append(1)

        elif against > against_th:
            preds.append(0)

        else:
            preds.append(2)

    return np.array(preds)

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred_balanced)

print(cm)

GLOVE+ATTENTION

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Layer
from tensorflow.keras import backend as K

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Layer
from tensorflow.keras import backend as K

class AttentionLayer(Layer):

    def __init__(self):
        super(AttentionLayer, self).__init__()

    def build(self, input_shape):
        self.W = self.add_weight(
            name="att_weight",
            shape=(input_shape[-1], 1),
            initializer="normal"
        )

        self.b = self.add_weight(
            name="att_bias",
            shape=(input_shape[1], 1),
            initializer="zeros"
        )

        super(AttentionLayer, self).build(input_shape)

    def call(self, x):

        e = K.tanh(K.dot(x, self.W) + self.b)

        a = K.softmax(e, axis=1)

        output = x * a

        return K.sum(output, axis=1)

In [ ]:
# from tensorflow.keras.layers import Input, Embedding, GRU, Dense, Dropout, SpatialDropout1D, Bidirectional
# from tensorflow.keras.models import Model

# input_layer = Input(shape=(max_sequence_length,))

# embedding = Embedding(
#     input_dim=vocab_size,
#     output_dim=embedding_dim,
#     weights=[embedding_matrix],
#     input_length=max_sequence_length,
#     trainable=True
# )(input_layer)

# x = SpatialDropout1D(0.4)(embedding)

# # Bidirectional GRU
# x = Bidirectional(GRU(128, return_sequences=True))(x)

# x = Dropout(0.4)(x)

# # Attention layer
# x = AttentionLayer()(x)

# x = Dropout(0.4)(x)

# output = Dense(3, activation="softmax")(x)

# model_gru_glove_attention = Model(inputs=input_layer, outputs=output)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout, SpatialDropout1D, Bidirectional
from tensorflow.keras.optimizers import Adam

max_sequence_length = X_train_padded.shape[1]   # ensures correct length

model_gru_glove_attention = Sequential()

model_gru_glove_attention.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        weights=[embedding_matrix],
        input_length=max_sequence_length,
        trainable=True
    )
)

model_gru_glove_attention.add(SpatialDropout1D(0.4))

model_gru_glove_attention.add(
    Bidirectional(
        GRU(96, return_sequences=True)
    )
)

model_gru_glove_attention.add(Dropout(0.4))

model_gru_glove_attention.add(
    Bidirectional(
        GRU(48)
    )
)

model_gru_glove_attention.add(Dense(3, activation="softmax"))

model_gru_glove_attention.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model_gru_glove_attention.summary()

In [ ]:
# from tensorflow.keras.optimizers import Adam

# model_glove_attention.compile(
#     optimizer=Adam(learning_rate=0.001),
#     loss="sparse_categorical_crossentropy",
#     metrics=["accuracy"]
# )
from tensorflow.keras.optimizers import Adam

model_gru_glove_attention.compile(
    optimizer=Adam(learning_rate=0.0005),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
# history_glove_attention = model_glove_attention.fit(

#     X_train_padded,
#     y_train,

#     validation_data=(X_val_padded, y_val),

#     epochs=25,
#     batch_size=32,

#     class_weight=class_weight_dict,

#     callbacks=[early_stop]
# )
# history_gru_glove_attention = model_gru_glove_attention.fit(

#     X_train_padded,
#     y_train,

#     validation_data=(X_val_padded, y_val),

#     epochs=25,
#     batch_size=32,

#     class_weight=class_weight_dict,

#     callbacks=[early_stop]
# )
import time

start_train = time.time()

history_gru_glove_attention = model_gru_glove_attention.fit(

    X_train_padded,
    y_train,

    validation_data=(X_val_padded, y_val),

    epochs=25,
    batch_size=32,

    class_weight=class_weight_dict,

    callbacks=[early_stop]
)

end_train = time.time()

train_time_hours = (end_train - start_train) / 3600

In [ ]:
start_infer = time.time()

y_pred_gru_attention = model_gru_glove_attention.predict(X_test_padded)

end_infer = time.time()

infer_time = end_infer - start_infer

In [ ]:
import numpy as np

y_pred_labels = np.argmax(y_pred_gru_attention, axis=1)

In [ ]:
import numpy as np

y_pred_labels = np.argmax(y_pred_gru_attention, axis=1)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy = accuracy_score(y_test, y_pred_labels)

precision = precision_score(
    y_test,
    y_pred_labels,
    average="macro"
)

recall = recall_score(
    y_test,
    y_pred_labels,
    average="macro"
)

f1 = f1_score(
    y_test,
    y_pred_labels,
    average="macro"
)

In [ ]:
params = model_gru_glove_attention.count_params() / 1e6

In [ ]:
print("\n===== GRU + GloVe + Attention RESULTS =====")

print("Accuracy:", round(accuracy,4))
print("Precision:", round(precision,4))
print("Recall:", round(recall,4))
print("F1 Score:", round(f1,4))

print("Train Time (h):", round(train_time_hours,3))
print("Infer Time (s):", round(infer_time,2))

print("Params (M):", round(params,2))

In [ ]:
import pandas as pd

table = pd.DataFrame({

    "Dataset": ["SemEval-A"],
    "Model": ["GRU + GloVe + Attention"],

    "Accuracy": [round(accuracy,4)],
    "Precision": [round(precision,4)],
    "Recall": [round(recall,4)],
    "F1-Score": [round(f1,4)],

    "Train Time (h)": [round(train_time_hours,3)],
    "Infer Time (s)": [round(infer_time,2)],

    "Params (M)": [round(params,2)]

})

print(table)

In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

y_pred_prob = model_gru_glove_attention.predict(X_test_padded)

y_test_bin = label_binarize(y_test, classes=[0,1,2])

n_classes = 3

fpr = {}
tpr = {}
roc_auc = {}

for i in range(n_classes):

    fpr[i], tpr[i], _ = roc_curve(
        y_test_bin[:, i],
        y_pred_prob[:, i]
    )

    roc_auc[i] = auc(fpr[i], tpr[i])

plt.figure(figsize=(6,6))

colors = ['blue','red','green']
labels = ['AGAINST','FAVOR','NONE']

for i in range(n_classes):

    plt.plot(
        fpr[i],
        tpr[i],
        color=colors[i],
        label=f"{labels[i]} (AUC = {roc_auc[i]:.3f})"
    )

plt.plot([0,1],[0,1],'k--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.title("ROC Curve – GRU + GloVe + Attention")

plt.legend()
plt.show()

In [ ]:
test_loss, test_acc = model_gru_glove_attention.evaluate(X_test_padded, y_test)

print("Test Accuracy:", test_acc)

In [ ]:
from sklearn.metrics import confusion_matrix

y_pred = model_gru_glove_attention.predict(X_test_padded)
y_pred_classes = np.argmax(y_pred, axis=1)

cm = confusion_matrix(y_test, y_pred_classes)

print(cm)

In [ ]:
y_probs = model_gru_glove_attention.predict(X_test_padded)

MINLSTM+GLOVE EMBEDDINGS

In [ ]:
pip install torch pandas numpy scikit-learn nltk matplotlib

In [ ]:
import pandas as pd
import numpy as np
import re
import time
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import LabelEncoder, label_binarize

from collections import Counter
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

import matplotlib.pyplot as plt

nltk.download('punkt')
nltk.download('stopwords')

In [ ]:
stop_words = set(stopwords.words("english"))

def clean_text(text):

    text = text.lower()

    text = re.sub(r"http\S+", "", text)

    text = re.sub(r"@\w+", "", text)

    text = re.sub(r"#", "", text)

    text = re.sub(r"[^a-zA-Z\s]", "", text)

    tokens = word_tokenize(text)

    tokens = [w for w in tokens if w not in stop_words]

    return " ".join(tokens)

In [ ]:
df = pd.read_excel("/content/train_data_semeval2016.xlsx")

df = df.dropna()

df["Tweet"] = df["Tweet"].apply(clean_text)

texts = df["Tweet"].values
labels = df["Stance"].values

In [ ]:
le = LabelEncoder()

labels = le.fit_transform(labels)

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    texts, labels, test_size=0.2, random_state=42)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42)

In [ ]:
counter = Counter()

for text in X_train:
    counter.update(word_tokenize(text))

vocab = {word:i+2 for i,(word,_) in enumerate(counter.most_common())}

vocab["<PAD>"] = 0
vocab["<UNK>"] = 1

In [ ]:
max_len = 80

def encode(text):

    tokens = word_tokenize(text)

    ids = [vocab.get(t,1) for t in tokens]

    if len(ids) < max_len:
        ids += [0]*(max_len-len(ids))
    else:
        ids = ids[:max_len]

    return ids

In [ ]:
import urllib.request
import zipfile
import os

url = "http://nlp.stanford.edu/data/glove.6B.zip"
zip_path = "glove.6B.zip"

print("Downloading GloVe embeddings...")

urllib.request.urlretrieve(url, zip_path)

print("Download complete.")

print("Extracting files...")

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall()

print("Extraction complete.")

In [ ]:
embedding_dim = 100

embeddings_index = {}

glove_path = "glove.6B.100d.txt"

with open(glove_path, encoding="utf8", errors="ignore") as f:

    for line in f:

        values = line.strip().split()

        if len(values) < embedding_dim + 1:
            continue

        word = values[0]

        vector = np.asarray(values[1:], dtype="float32")

        embeddings_index[word] = vector

print("Total GloVe vectors loaded:", len(embeddings_index))

In [ ]:
vocab_size = len(vocab)

embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, idx in vocab.items():

    vector = embeddings_index.get(word)

    if vector is not None:
        embedding_matrix[idx] = vector

In [ ]:
class TextDataset(Dataset):

    def __init__(self,texts,labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self,idx):

        x = torch.tensor(encode(self.texts[idx]),dtype=torch.long)

        y = torch.tensor(self.labels[idx],dtype=torch.long)

        return x,y

In [ ]:
train_loader = DataLoader(TextDataset(X_train,y_train),batch_size=32,shuffle=True)

val_loader = DataLoader(TextDataset(X_val,y_val),batch_size=32)

test_loader = DataLoader(TextDataset(X_test,y_test),batch_size=32)

In [ ]:
class MiniLSTM(nn.Module):

    def __init__(self,vocab_size,embedding_dim,hidden_dim,num_classes):

        super(MiniLSTM,self).__init__()

        self.embedding = nn.Embedding(vocab_size,embedding_dim)

        self.embedding.weight.data.copy_(torch.tensor(embedding_matrix))

        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(0.4)

        self.fc = nn.Linear(hidden_dim*2,num_classes)

    def forward(self,x):

        x = self.embedding(x)

        output,(h,c) = self.lstm(x)

        h = torch.cat((h[-2],h[-1]),dim=1)

        h = self.dropout(h)

        out = self.fc(h)

        return out

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MiniLSTM(
    vocab_size,
    embedding_dim=100,
    hidden_dim=128,
    num_classes=len(np.unique(labels))
).to(device)

In [ ]:
total_params = sum(p.numel() for p in model.parameters())

params_millions = total_params / 1e6

print("Total Parameters (Millions):", params_millions)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(),lr=0.001)

In [ ]:
epochs = 12

train_start = time.time()

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for inputs,labels in train_loader:

        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)

        loss = criterion(outputs,labels)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)

        optimizer.step()

        total_loss += loss.item()

    print("Epoch:",epoch+1,"Loss:",total_loss/len(train_loader))

train_end = time.time()

train_time_hours = (train_end - train_start)/3600

print("Training Time (hours):",train_time_hours)

In [ ]:
infer_start = time.time()

model.eval()

preds = []
true = []
probs = []

with torch.no_grad():

    for inputs,labels in test_loader:

        inputs = inputs.to(device)

        outputs = model(inputs)

        prob = torch.softmax(outputs,dim=1).cpu().numpy()

        pred = np.argmax(prob,axis=1)

        preds.extend(pred)
        true.extend(labels.numpy())
        probs.extend(prob)

infer_end = time.time()

infer_time = infer_end - infer_start

print("Inference Time (seconds):",infer_time)

In [ ]:
acc = accuracy_score(true,preds)

prec = precision_score(true,preds,average="macro")

rec = recall_score(true,preds,average="macro")

f1 = f1_score(true,preds,average="macro")

print("Accuracy:",acc)
print("Precision:",prec)
print("Recall:",rec)
print("F1 Score:",f1)

In [ ]:
true_bin = label_binarize(true,classes=list(range(len(np.unique(true)))))

probs = np.array(probs)

fpr = {}
tpr = {}
roc_auc = {}

for i in range(true_bin.shape[1]):

    fpr[i], tpr[i], _ = roc_curve(true_bin[:,i],probs[:,i])

    roc_auc[i] = auc(fpr[i],tpr[i])

plt.figure()

for i in range(true_bin.shape[1]):

    plt.plot(
        fpr[i],
        tpr[i],
        label="Class {} (AUC={:.2f})".format(i,roc_auc[i])
    )

plt.plot([0,1],[0,1],'k--')

plt.xlabel("False Positive Rate")

plt.ylabel("True Positive Rate")

plt.title("ROC Curve - Mini LSTM + GloVe")

plt.legend()

plt.show()

MINLSTM+GLOVE+ATTENTION

In [ ]:
pip install torch pandas numpy scikit-learn nltk matplotlib

In [ ]:
import pandas as pd
import numpy as np
import re
import time
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_curve, auc

from collections import Counter
import matplotlib.pyplot as plt
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

import urllib.request
import zipfile
import os

nltk.download("punkt")
nltk.download("stopwords")

In [ ]:
stop_words = set(stopwords.words("english"))

def clean_text(text):

    text = text.lower()

    text = re.sub(r"http\S+", "", text)

    text = re.sub(r"@\w+", "", text)

    text = re.sub(r"#", "", text)

    text = re.sub(r"[^a-zA-Z\s]", "", text)

    tokens = word_tokenize(text)

    tokens = [w for w in tokens if w not in stop_words]

    return " ".join(tokens)

In [ ]:
df = pd.read_excel("train_data_semeval2016.xlsx")

df = df.dropna()

df["Tweet"] = df["Tweet"].apply(clean_text)

texts = df["Tweet"].values
labels = df["Stance"].values

In [ ]:
le = LabelEncoder()

labels = le.fit_transform(labels)

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    texts, labels, test_size=0.2, random_state=42)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42)

In [ ]:
counter = Counter()

for text in X_train:
    counter.update(word_tokenize(text))

vocab = {word:i+2 for i,(word,_) in enumerate(counter.most_common())}

vocab["<PAD>"] = 0
vocab["<UNK>"] = 1

In [ ]:
max_len = 80

def encode(text):

    tokens = word_tokenize(text)

    ids = [vocab.get(t,1) for t in tokens]

    if len(ids) < max_len:
        ids += [0]*(max_len-len(ids))
    else:
        ids = ids[:max_len]

    return ids

In [ ]:
if not os.path.exists("glove.6B.100d.txt"):

    print("Downloading GloVe...")

    urllib.request.urlretrieve(
        "http://nlp.stanford.edu/data/glove.6B.zip",
        "glove.6B.zip"
    )

    with zipfile.ZipFile("glove.6B.zip","r") as zip_ref:
        zip_ref.extractall()

    print("GloVe downloaded")

In [ ]:
embedding_dim = 100

embeddings_index = {}

with open("glove.6B.100d.txt",encoding="utf8",errors="ignore") as f:

    for line in f:

        values = line.strip().split()

        if len(values) < embedding_dim + 1:
            continue

        word = values[0]

        vector = np.asarray(values[1:],dtype="float32")

        embeddings_index[word] = vector

print("Loaded GloVe vectors:",len(embeddings_index))

In [ ]:
vocab_size = len(vocab)

embedding_matrix = np.zeros((vocab_size,embedding_dim))

for word,idx in vocab.items():

    vector = embeddings_index.get(word)

    if vector is not None:
        embedding_matrix[idx] = vector

In [ ]:
class TextDataset(Dataset):

    def __init__(self,texts,labels):

        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self,idx):

        x = torch.tensor(encode(self.texts[idx]),dtype=torch.long)

        y = torch.tensor(self.labels[idx],dtype=torch.long)

        return x,y

In [ ]:
train_loader = DataLoader(TextDataset(X_train,y_train),batch_size=32,shuffle=True)

val_loader = DataLoader(TextDataset(X_val,y_val),batch_size=32)

test_loader = DataLoader(TextDataset(X_test,y_test),batch_size=32)

In [ ]:
class MiniLSTMAttention(nn.Module):

    def __init__(self,vocab_size,embedding_dim,hidden_dim,num_classes):

        super(MiniLSTMAttention,self).__init__()

        self.embedding = nn.Embedding(vocab_size,embedding_dim)

        self.embedding.weight.data.copy_(torch.tensor(embedding_matrix))

        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.attention = nn.Linear(hidden_dim*2,1)

        self.dropout = nn.Dropout(0.4)

        self.fc = nn.Linear(hidden_dim*2,num_classes)

    def forward(self,x):

        x = self.embedding(x)

        lstm_out,_ = self.lstm(x)

        attn_weights = torch.softmax(
            self.attention(lstm_out),
            dim=1
        )

        context = torch.sum(attn_weights * lstm_out,dim=1)

        context = self.dropout(context)

        out = self.fc(context)

        return out

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MiniLSTMAttention(
    vocab_size,
    embedding_dim=100,
    hidden_dim=128,
    num_classes=len(np.unique(labels))
).to(device)

In [ ]:
total_params = sum(p.numel() for p in model.parameters())

params_millions = total_params / 1e6

print("Total Parameters (Millions):",params_millions)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(),lr=0.001)

In [ ]:
epochs = 12

train_start = time.time()

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for inputs,labels in train_loader:

        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)

        loss = criterion(outputs,labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print("Epoch",epoch+1,"Loss:",total_loss/len(train_loader))

train_end = time.time()

train_time_hours = (train_end-train_start)/3600

print("Training Time (hours):",train_time_hours)

In [ ]:
infer_start = time.time()

model.eval()

preds = []
true = []
probs = []

with torch.no_grad():

    for inputs,labels in test_loader:

        inputs = inputs.to(device)

        outputs = model(inputs)

        prob = torch.softmax(outputs,dim=1).cpu().numpy()

        pred = np.argmax(prob,axis=1)

        preds.extend(pred)
        true.extend(labels.numpy())
        probs.extend(prob)

infer_end = time.time()

infer_time = infer_end - infer_start

print("Inference Time (seconds):",infer_time)

In [ ]:
acc = accuracy_score(true,preds)

prec = precision_score(true,preds,average="macro")

rec = recall_score(true,preds,average="macro")

f1 = f1_score(true,preds,average="macro")

print("Accuracy:",acc)
print("Precision:",prec)
print("Recall:",rec)
print("F1 Score:",f1)

In [ ]:
true_bin = label_binarize(true,classes=list(range(len(np.unique(true)))))

probs = np.array(probs)

fpr = {}
tpr = {}
roc_auc = {}

for i in range(true_bin.shape[1]):

    fpr[i],tpr[i],_ = roc_curve(true_bin[:,i],probs[:,i])

    roc_auc[i] = auc(fpr[i],tpr[i])

plt.figure()

for i in range(true_bin.shape[1]):

    plt.plot(
        fpr[i],
        tpr[i],
        label="Class {} (AUC={:.2f})".format(i,roc_auc[i])
    )

plt.plot([0,1],[0,1],'k--')

plt.xlabel("False Positive Rate")

plt.ylabel("True Positive Rate")

plt.title("ROC Curve - MiniLSTM + GloVe + Attention")

plt.legend()

plt.show()

MINGRU+GLOVE EMBEDDINGS

In [ ]:
pip install torch pandas numpy scikit-learn nltk matplotlib openpyxl

In [ ]:
import pandas as pd
import numpy as np
import re
import time
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_curve, auc

from collections import Counter
import matplotlib.pyplot as plt
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

import urllib.request
import zipfile
import os

nltk.download("punkt")
nltk.download("stopwords")

In [ ]:
stop_words = set(stopwords.words("english"))

def clean_text(text):

    text = text.lower()

    text = re.sub(r"http\S+", "", text)

    text = re.sub(r"@\w+", "", text)

    text = re.sub(r"#", "", text)

    text = re.sub(r"[^a-zA-Z\s]", "", text)

    tokens = word_tokenize(text)

    tokens = [w for w in tokens if w not in stop_words]

    return " ".join(tokens)

In [ ]:
df = pd.read_excel("train_data_semeval2016.xlsx")

df = df.dropna()

df["Tweet"] = df["Tweet"].apply(clean_text)

texts = df["Tweet"].values
labels = df["Stance"].values

In [ ]:
le = LabelEncoder()

labels = le.fit_transform(labels)

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    texts, labels, test_size=0.2, random_state=42)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42)

In [ ]:
counter = Counter()

for text in X_train:
    counter.update(word_tokenize(text))

vocab = {word:i+2 for i,(word,_) in enumerate(counter.most_common())}

vocab["<PAD>"] = 0
vocab["<UNK>"] = 1

In [ ]:
max_len = 80

def encode(text):

    tokens = word_tokenize(text)

    ids = [vocab.get(t,1) for t in tokens]

    if len(ids) < max_len:
        ids += [0]*(max_len-len(ids))
    else:
        ids = ids[:max_len]

    return ids

In [ ]:
if not os.path.exists("glove.6B.100d.txt"):

    print("Downloading GloVe...")

    urllib.request.urlretrieve(
        "http://nlp.stanford.edu/data/glove.6B.zip",
        "glove.6B.zip"
    )

    with zipfile.ZipFile("glove.6B.zip","r") as zip_ref:
        zip_ref.extractall()

    print("GloVe downloaded")

In [ ]:
embedding_dim = 100

embeddings_index = {}

with open("glove.6B.100d.txt",encoding="utf8",errors="ignore") as f:

    for line in f:

        values = line.strip().split()

        if len(values) < embedding_dim + 1:
            continue

        word = values[0]

        vector = np.asarray(values[1:],dtype="float32")

        embeddings_index[word] = vector

print("Loaded GloVe vectors:",len(embeddings_index))

In [ ]:
vocab_size = len(vocab)

embedding_matrix = np.zeros((vocab_size,embedding_dim))

for word,idx in vocab.items():

    vector = embeddings_index.get(word)

    if vector is not None:
        embedding_matrix[idx] = vector

In [ ]:
class TextDataset(Dataset):

    def __init__(self,texts,labels):

        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self,idx):

        x = torch.tensor(encode(self.texts[idx]),dtype=torch.long)

        y = torch.tensor(self.labels[idx],dtype=torch.long)

        return x,y

In [ ]:
train_loader = DataLoader(TextDataset(X_train,y_train),batch_size=32,shuffle=True)

val_loader = DataLoader(TextDataset(X_val,y_val),batch_size=32)

test_loader = DataLoader(TextDataset(X_test,y_test),batch_size=32)

In [ ]:
class MiniGRU(nn.Module):

    def __init__(self,vocab_size,embedding_dim,hidden_dim,num_classes):

        super(MiniGRU,self).__init__()

        self.embedding = nn.Embedding(vocab_size,embedding_dim)

        self.embedding.weight.data.copy_(torch.tensor(embedding_matrix))

        self.gru = nn.GRU(
            embedding_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(0.4)

        self.fc = nn.Linear(hidden_dim*2,num_classes)

    def forward(self,x):

        x = self.embedding(x)

        output,h = self.gru(x)

        h = torch.cat((h[-2],h[-1]),dim=1)

        h = self.dropout(h)

        out = self.fc(h)

        return out

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MiniGRU(
    vocab_size,
    embedding_dim=100,
    hidden_dim=128,
    num_classes=len(np.unique(labels))
).to(device)

In [ ]:
total_params = sum(p.numel() for p in model.parameters())

params_millions = total_params / 1e6

print("Total Parameters (Millions):",params_millions)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(),lr=0.001)

In [ ]:
epochs = 12

train_start = time.time()

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for inputs,labels in train_loader:

        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)

        loss = criterion(outputs,labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print("Epoch",epoch+1,"Loss:",total_loss/len(train_loader))

train_end = time.time()

train_time_hours = (train_end-train_start)/3600

print("Training Time (hours):",train_time_hours)

In [ ]:
infer_start = time.time()

model.eval()

preds = []
true = []
probs = []

with torch.no_grad():

    for inputs,labels in test_loader:

        inputs = inputs.to(device)

        outputs = model(inputs)

        prob = torch.softmax(outputs,dim=1).cpu().numpy()

        pred = np.argmax(prob,axis=1)

        preds.extend(pred)
        true.extend(labels.numpy())
        probs.extend(prob)

infer_end = time.time()

infer_time = infer_end - infer_start

print("Inference Time (seconds):",infer_time)

In [ ]:
acc = accuracy_score(true,preds)

prec = precision_score(true,preds,average="macro")

rec = recall_score(true,preds,average="macro")

f1 = f1_score(true,preds,average="macro")

print("Accuracy:",acc)
print("Precision:",prec)
print("Recall:",rec)
print("F1 Score:",f1)

In [ ]:
true_bin = label_binarize(true,classes=list(range(len(np.unique(true)))))

probs = np.array(probs)

fpr = {}
tpr = {}
roc_auc = {}

for i in range(true_bin.shape[1]):

    fpr[i],tpr[i],_ = roc_curve(true_bin[:,i],probs[:,i])

    roc_auc[i] = auc(fpr[i],tpr[i])

plt.figure()

for i in range(true_bin.shape[1]):

    plt.plot(
        fpr[i],
        tpr[i],
        label="Class {} (AUC={:.2f})".format(i,roc_auc[i])
    )

plt.plot([0,1],[0,1],'k--')

plt.xlabel("False Positive Rate")

plt.ylabel("True Positive Rate")

plt.title("ROC Curve - MiniGRU + GloVe")

plt.legend()

plt.show()

MINGRU+GLOVE+ATTENTION

In [ ]:
pip install torch pandas numpy scikit-learn nltk matplotlib openpyxl

In [ ]:
import pandas as pd
import numpy as np
import re
import time
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_curve, auc

from collections import Counter
import matplotlib.pyplot as plt

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

import urllib.request
import zipfile
import os

nltk.download("punkt")
nltk.download("stopwords")

In [ ]:
stop_words = set(stopwords.words("english"))

def clean_text(text):

    text = text.lower()

    text = re.sub(r"http\S+", "", text)

    text = re.sub(r"@\w+", "", text)

    text = re.sub(r"#", "", text)

    text = re.sub(r"[^a-zA-Z\s]", "", text)

    tokens = word_tokenize(text)

    tokens = [w for w in tokens if w not in stop_words]

    return " ".join(tokens)

In [ ]:
df = pd.read_excel("train_data_semeval2016.xlsx")

df = df.dropna()

df["Tweet"] = df["Tweet"].apply(clean_text)

texts = df["Tweet"].values
labels = df["Stance"].values

In [ ]:
le = LabelEncoder()

labels = le.fit_transform(labels)

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    texts, labels, test_size=0.2, random_state=42)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42)

In [ ]:
counter = Counter()

for text in X_train:
    counter.update(word_tokenize(text))

vocab = {word:i+2 for i,(word,_) in enumerate(counter.most_common())}

vocab["<PAD>"] = 0
vocab["<UNK>"] = 1

In [ ]:
max_len = 80

def encode(text):

    tokens = word_tokenize(text)

    ids = [vocab.get(t,1) for t in tokens]

    if len(ids) < max_len:
        ids += [0]*(max_len-len(ids))
    else:
        ids = ids[:max_len]

    return ids

In [ ]:
if not os.path.exists("glove.6B.100d.txt"):

    print("Downloading GloVe...")

    urllib.request.urlretrieve(
        "http://nlp.stanford.edu/data/glove.6B.zip",
        "glove.6B.zip"
    )

    with zipfile.ZipFile("glove.6B.zip","r") as zip_ref:
        zip_ref.extractall()

    print("GloVe downloaded")

In [ ]:
embedding_dim = 100

embeddings_index = {}

with open("glove.6B.100d.txt",encoding="utf8",errors="ignore") as f:

    for line in f:

        values = line.strip().split()

        if len(values) < embedding_dim + 1:
            continue

        word = values[0]

        vector = np.asarray(values[1:],dtype="float32")

        embeddings_index[word] = vector

print("Loaded GloVe vectors:",len(embeddings_index))

In [ ]:
vocab_size = len(vocab)

embedding_matrix = np.zeros((vocab_size,embedding_dim))

for word,idx in vocab.items():

    vector = embeddings_index.get(word)

    if vector is not None:
        embedding_matrix[idx] = vector


In [ ]:
class TextDataset(Dataset):

    def __init__(self,texts,labels):

        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self,idx):

        x = torch.tensor(encode(self.texts[idx]),dtype=torch.long)

        y = torch.tensor(self.labels[idx],dtype=torch.long)

        return x,y

In [ ]:
train_loader = DataLoader(TextDataset(X_train,y_train),batch_size=32,shuffle=True)

val_loader = DataLoader(TextDataset(X_val,y_val),batch_size=32)

test_loader = DataLoader(TextDataset(X_test,y_test),batch_size=32)

In [ ]:
class MiniGRUAttention(nn.Module):

    def __init__(self,vocab_size,embedding_dim,hidden_dim,num_classes):

        super(MiniGRUAttention,self).__init__()

        self.embedding = nn.Embedding(vocab_size,embedding_dim)

        self.embedding.weight.data.copy_(torch.tensor(embedding_matrix))

        self.gru = nn.GRU(
            embedding_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.attention = nn.Linear(hidden_dim*2,1)

        self.dropout = nn.Dropout(0.4)

        self.fc = nn.Linear(hidden_dim*2,num_classes)

    def forward(self,x):

        x = self.embedding(x)

        gru_out,_ = self.gru(x)

        attn_weights = torch.softmax(
            self.attention(gru_out),
            dim=1
        )

        context = torch.sum(attn_weights * gru_out,dim=1)

        context = self.dropout(context)

        out = self.fc(context)

        return out

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MiniGRUAttention(
    vocab_size,
    embedding_dim=100,
    hidden_dim=128,
    num_classes=len(np.unique(labels))
).to(device)

In [ ]:
total_params = sum(p.numel() for p in model.parameters())

params_millions = total_params / 1e6

print("Total Parameters (Millions):",params_millions)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(),lr=0.001)

In [ ]:
epochs = 12

train_start = time.time()

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for inputs,labels in train_loader:

        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)

        loss = criterion(outputs,labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print("Epoch",epoch+1,"Loss:",total_loss/len(train_loader))

train_end = time.time()

train_time_hours = (train_end-train_start)/3600

print("Training Time (hours):",train_time_hours)

In [ ]:
infer_start = time.time()

model.eval()

preds = []
true = []
probs = []

with torch.no_grad():

    for inputs,labels in test_loader:

        inputs = inputs.to(device)

        outputs = model(inputs)

        prob = torch.softmax(outputs,dim=1).cpu().numpy()

        pred = np.argmax(prob,axis=1)

        preds.extend(pred)
        true.extend(labels.numpy())
        probs.extend(prob)

infer_end = time.time()

infer_time = infer_end - infer_start

print("Inference Time (seconds):",infer_time)

In [ ]:
acc = accuracy_score(true,preds)

prec = precision_score(true,preds,average="macro")

rec = recall_score(true,preds,average="macro")

f1 = f1_score(true,preds,average="macro")

print("Accuracy:",acc)
print("Precision:",prec)
print("Recall:",rec)
print("F1 Score:",f1)

In [ ]:
true_bin = label_binarize(true,classes=list(range(len(np.unique(true)))))

probs = np.array(probs)

fpr = {}
tpr = {}
roc_auc = {}

for i in range(true_bin.shape[1]):

    fpr[i],tpr[i],_ = roc_curve(true_bin[:,i],probs[:,i])

    roc_auc[i] = auc(fpr[i],tpr[i])

plt.figure()

for i in range(true_bin.shape[1]):

    plt.plot(
        fpr[i],
        tpr[i],
        label="Class {} (AUC={:.2f})".format(i,roc_auc[i])
    )

plt.plot([0,1],[0,1],'k--')

plt.xlabel("False Positive Rate")

plt.ylabel("True Positive Rate")

plt.title("ROC Curve - MiniGRU + GloVe + Attention")

plt.legend()

plt.show()

BERT FOR SEM EVAL TASK B

In [ ]:
pip install pandas scikit-learn transformers torch nltk

In [ ]:
import pandas as pd
import numpy as np
import re
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import LabelEncoder
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW # Corrected import path for AdamW
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

In [ ]:
stop_words = set(stopwords.words("english"))

def clean_text(text):

    text = text.lower()

    text = re.sub(r"http\S+", "", text)

    text = re.sub(r"@\w+", "", text)

    text = re.sub(r"#", "", text)

    text = re.sub(r"[^a-zA-Z\s]", "", text)

    words = text.split()

    words = [w for w in words if w not in stop_words]

    return " ".join(words)

In [ ]:
df = pd.read_excel("train_data_semeval2016.xlsx")

df = df.dropna()

df["text"] = df["Tweet"].apply(clean_text)

texts = df["text"].values
labels = df["Stance"].values

In [ ]:
# This cell was redundant and was causing labels and texts to be overwritten with pandas Series. Content removed.

In [ ]:
le = LabelEncoder()

labels = le.fit_transform(labels)

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    texts, labels, test_size=0.2, random_state=42)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42)

X_train = list(X_train)
X_val = list(X_val)
X_test = list(X_test)
y_train = list(y_train)
y_val = list(y_val)
y_test = list(y_test)

In [ ]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
class BertDataset(Dataset):

    def __init__(self, texts, labels, tokenizer, max_len=128):

        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "labels": torch.tensor(label, dtype=torch.long)
        }

In [ ]:
train_loader = DataLoader(
    BertDataset(X_train,y_train,tokenizer),
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    BertDataset(X_val,y_val,tokenizer),
    batch_size=16
)

test_loader = DataLoader(
    BertDataset(X_test,y_test,tokenizer),
    batch_size=16
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(np.unique(labels))
)

model = model.to(device)

In [ ]:
params = sum(p.numel() for p in bart_model.parameters())

params_millions = params / 1e6

print("Total Parameters (Millions):", params_millions)

In [ ]:
optimizer = AdamW(model.parameters(), lr=2e-5)

criterion = nn.CrossEntropyLoss()

In [ ]:
epochs = 4

train_start = time.time()

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for batch in train_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)

        optimizer.step()

        total_loss += loss.item()

    print("Epoch:", epoch+1, "Loss:", total_loss/len(train_loader))

train_end = time.time()

train_time_hours = (train_end - train_start) / 3600

print("Training Time (hours):", train_time_hours)

In [ ]:
infer_start = time.time()

model.eval()

preds = []
true = []

with torch.no_grad():

    for batch in test_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        predictions = torch.argmax(outputs.logits, dim=1).cpu().numpy()

        preds.extend(predictions)
        true.extend(batch["labels"].numpy())

infer_end = time.time()

infer_time = infer_end - infer_start

print("Inference Time (seconds):", infer_time)

In [ ]:
acc = accuracy_score(true, preds)

prec = precision_score(true, preds, average="macro")

rec = recall_score(true, preds, average="macro")

f1 = f1_score(true, preds, average="macro")

print("Accuracy:", acc)
print("Precision:", prec)
print("Recall:", rec)
print("F1 Score:", f1)

In [ ]:
params = sum(p.numel() for p in model.parameters())

params_millions = params / 1e6

print(f"Total Parameters (Millions): {params_millions:.2f}")

BART


In [ ]:
pip install transformers torch pandas scikit-learn nltk matplotlib openpyxl tqdm

In [ ]:
import pandas as pd
import numpy as np
import re
import time
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import BartTokenizer, BartForSequenceClassification
from torch.optim import AdamW

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_curve, auc

import matplotlib.pyplot as plt
from tqdm import tqdm

import nltk
from nltk.corpus import stopwords
nltk.download("stopwords")

In [ ]:
stop_words = set(stopwords.words("english"))

def clean_text(text):

    text = text.lower()

    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)

    words = text.split()

    words = [w for w in words if w not in stop_words]

    return " ".join(words)

In [ ]:
df = pd.read_excel("train_data_semeval2016.xlsx")

df = df.dropna()

df["Tweet"] = df["Tweet"].apply(clean_text)

texts = df["Tweet"].values
labels = df["Stance"].values

In [ ]:
le = LabelEncoder()

labels = le.fit_transform(labels)

num_labels = len(np.unique(labels))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42
)

In [ ]:
tokenizer = BartTokenizer.from_pretrained("facebook/bart-base")

In [ ]:
class StanceDataset(Dataset):

    def __init__(self,texts,labels):

        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self,idx):

        text = str(self.texts[idx])

        encoding = tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels": torch.tensor(self.labels[idx])
        }

In [ ]:
train_loader = DataLoader(
    StanceDataset(X_train,y_train),
    batch_size=16,
    shuffle=True
)

test_loader = DataLoader(
    StanceDataset(X_test,y_test),
    batch_size=16
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

bart_model = BartForSequenceClassification.from_pretrained(
    "facebook/bart-base",
    num_labels=num_labels
)

bart_model = bart_model.to(device)

In [ ]:
optimizer = AdamW(bart_model.parameters(), lr=2e-5)

In [ ]:
params = sum(p.numel() for p in bart_model.parameters()) / 1e6

print("Parameters (Millions):", round(params,2))

In [ ]:
EPOCHS = 4

train_start = time.time()

for epoch in range(EPOCHS):

    bart_model.train()

    total_loss = 0

    loop = tqdm(train_loader)

    for batch in loop:

        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = bart_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        total_loss += loss.item()

        loss.backward()

        torch.nn.utils.clip_grad_norm_(bart_model.parameters(),1.0)

        optimizer.step()

        loop.set_description(f"Epoch {epoch+1}")
        loop.set_postfix(loss=loss.item())

    print("Epoch Loss:", total_loss/len(train_loader))

train_end = time.time()

train_time_hours = (train_end - train_start)/3600

In [ ]:
bart_model.eval()

predictions = []
true_labels = []
probs = []

infer_start = time.time()

with torch.no_grad():

    for batch in tqdm(test_loader):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        outputs = bart_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits

        prob = torch.softmax(logits,dim=1).cpu().numpy()

        preds = np.argmax(prob,axis=1)

        predictions.extend(preds)
        true_labels.extend(batch["labels"].numpy())
        probs.extend(prob)

infer_end = time.time()

inference_time = infer_end - infer_start

In [ ]:
accuracy = accuracy_score(true_labels,predictions)

precision = precision_score(true_labels,predictions,average="macro")

recall = recall_score(true_labels,predictions,average="macro")

f1 = f1_score(true_labels,predictions,average="macro")

print("\n===== BART RESULTS =====")

print("Accuracy:", round(accuracy,4))
print("Precision:", round(precision,4))
print("Recall:", round(recall,4))
print("F1 Score:", round(f1,4))

print("Train Time (h):", round(train_time_hours,3))
print("Inference Time (s):", round(inference_time,2))

print("Parameters (M):", round(params,2))

In [ ]:
true_bin = label_binarize(true_labels, classes=list(range(num_labels)))

probs = np.array(probs)

fpr = {}
tpr = {}
roc_auc = {}

for i in range(num_labels):

    fpr[i],tpr[i],_ = roc_curve(true_bin[:,i],probs[:,i])

    roc_auc[i] = auc(fpr[i],tpr[i])

plt.figure()

for i in range(num_labels):

    plt.plot(
        fpr[i],
        tpr[i],
        label="Class {} (AUC={:.2f})".format(i,roc_auc[i])
    )

plt.plot([0,1],[0,1],'k--')

plt.xlabel("False Positive Rate")

plt.ylabel("True Positive Rate")

plt.title("ROC Curve - BART")

plt.legend()

plt.show()